In [ ]:
# 1. Install system utilities fast
!apt-get install -y --no-install-recommends poppler-utils tesseract-ocr

# 2. Fast install with pre-built binary wheels (no dependency loops)
!pip install --prefer-binary \
    gradio \
    langgraph \
    langchain-google-genai \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    rank_bm25 \
    pdfplumber \
    pdf2image \
    pillow \
    pydantic

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 11 not upgraded.


In [ ]:
!apt-get install -y --no-install-recommends poppler-utils tesseract-ocr
!pip install --prefer-binary -q \
    gradio \
    langgraph \
    langchain-google-genai \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    pdfplumber \
    pdf2image \
    pillow \
    pydantic

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 11 not upgraded.


In [ ]:
!apt-get install -y --no-install-recommends poppler-utils tesseract-ocr
!pip install --prefer-binary gradio langgraph langchain-google-genai langchain-core langchain-community langchain-text-splitters faiss-cpu rank_bm25 pdfplumber pdf2image pillow pydantic

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 11 not upgraded.


In [ ]:
import os
import time
import json
import base64
from io import BytesIO
from typing import TypedDict, Optional, List, Dict, Any

import gradio as gr
from PIL import Image
import pdfplumber
from pdf2image import convert_from_path

from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langgraph.graph import StateGraph, START, END

# ==============================================================================
# 1. STATE DEFINITIONS & CACHE
# ==============================================================================
class AgentState(TypedDict):
    file_path: str
    is_scanned: bool
    clean_markdown: str
    metadata: Dict[str, Any]
    vectorstore: Optional[object]
    bm25_retriever: Optional[object]
    latency: float

active_session = {
    "vectorstore": None,
    "bm25_retriever": None,
    "api_key": "",
    "model_name": ""
}

# ==============================================================================
# 2. OPTIMIZED MULTI-AGENT PIPELINE
# ==============================================================================
def build_pipeline(model_name: str, api_key: str):
    llm = ChatGoogleGenerativeAI(model=model_name, google_api_key=api_key, temperature=0)

    def process_document(state: AgentState) -> AgentState:
        t0 = time.time()
        file_path = state["file_path"]
        extracted_text = ""
        page_count = 0

        # Step A: Fast Text Extraction Check
        try:
            with pdfplumber.open(file_path) as pdf:
                page_count = len(pdf.pages)
                for p in pdf.pages[:5]:
                    t = p.extract_text()
                    if t:
                        extracted_text += t + "\n"
        except Exception:
            extracted_text = ""

        is_scanned = (len(extracted_text.strip()) / max(page_count, 1)) < 50

        # Step B: Vision Fallback if Scanned, else Clean Direct Text
        if is_scanned:
            images = convert_from_path(file_path, first_page=1, last_page=3, dpi=100)
            image_contents = []
            for img in images:
                buf = BytesIO()
                img.save(buf, format="JPEG", quality=70)
                img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
                image_contents.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
                })

            prompt_msg = HumanMessage(content=[
                {"type": "text", "text": "Extract all text cleanly as structured Markdown. Output ONLY the document content with proper tables and headers."},
                *image_contents
            ])
            extracted_markdown = llm.invoke([prompt_msg]).content
        else:
            prompt = f"Clean minor OCR typos and structure this document text into clean Markdown with tables and headers:\n\n{extracted_text}"
            extracted_markdown = llm.invoke(prompt).content

        # Step C: Metadata Extraction
        meta_prompt = f"""Summarize this document briefly in JSON format:
{{"title": "Document Title", "type": "Document Category", "summary": "2-sentence summary"}}

CONTENT:
{extracted_markdown[:1500]}"""

        try:
            meta_res = llm.invoke(meta_prompt).content
            clean_json = meta_res[meta_res.find("{"):meta_res.rfind("}")+1]
            metadata = json.loads(clean_json)
        except Exception:
            metadata = {"title": "Document", "type": "General", "summary": "Processed document."}

        # Step D: Indexing (Vector + BM25)
        splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        docs = splitter.create_documents([extracted_markdown])

        embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview", google_api_key=api_key)
        vectorstore = FAISS.from_documents(docs, embeddings)

        bm25_retriever = BM25Retriever.from_documents(docs)
        bm25_retriever.k = 3

        return {
            "file_path": file_path,
            "is_scanned": is_scanned,
            "clean_markdown": extracted_markdown,
            "metadata": metadata,
            "vectorstore": vectorstore,
            "bm25_retriever": bm25_retriever,
            "latency": round(time.time() - t0, 2)
        }

    workflow = StateGraph(AgentState)
    workflow.add_node("process_all", process_document)
    workflow.add_edge(START, "process_all")
    workflow.add_edge("process_all", END)

    return workflow.compile(), llm

# ==============================================================================
# 3. UI HANDLERS
# ==============================================================================
def process_pdf(file, api_key, model_name):
    if not api_key:
        return "⚠️ Please enter your Gemini API Key.", "{}", "Missing Key"
    if not file:
        return "⚠️ Please upload a PDF.", "{}", "No File"

    agent_app, _ = build_pipeline(model_name, api_key)

    init_state = {
        "file_path": file.name,
        "is_scanned": False,
        "clean_markdown": "",
        "metadata": {},
        "vectorstore": None,
        "bm25_retriever": None,
        "latency": 0.0
    }

    res = agent_app.invoke(init_state)

    active_session["vectorstore"] = res["vectorstore"]
    active_session["bm25_retriever"] = res["bm25_retriever"]
    active_session["api_key"] = api_key
    active_session["model_name"] = model_name

    status = f"⚡ Finished in {res['latency']}s | Mode: {'Vision OCR' if res['is_scanned'] else 'Direct Parser'}"
    return res["clean_markdown"], json.dumps(res["metadata"], indent=2), status


def answer_rag_query(query):
    if not query or active_session["vectorstore"] is None:
        return "⚠️ Upload and process a PDF first."

    llm = ChatGoogleGenerativeAI(
        model=active_session["model_name"],
        google_api_key=active_session["api_key"],
        temperature=0
    )

    # Hybrid Search: Combine dense FAISS vector results + BM25 keyword results
    dense_docs = active_session["vectorstore"].similarity_search(query, k=3)
    sparse_docs = active_session["bm25_retriever"].invoke(query)

    # Deduplicate retrieved context
    seen_texts = set()
    combined_docs = []
    for d in dense_docs + sparse_docs:
        if d.page_content not in seen_texts:
            combined_docs.append(d.page_content)
            seen_texts.add(d.page_content)

    context_str = "\n\n".join(combined_docs[:4])

    prompt = f"""Answer the question strictly using the provided context. If the answer is not in the context, state 'Information not found in document.'

CONTEXT:
{context_str}

QUESTION:
{query}"""

    return llm.invoke(prompt).content

# ==============================================================================
# 4. GRADIO INTERFACE
# ==============================================================================
with gr.Blocks(theme=gr.themes.Soft(), title="DocuMind AI") as demo:
    gr.Markdown("# ⚡ DocuMind AI — Agentic OCR & Hybrid RAG")

    with gr.Row():
        api_key_input = gr.Textbox(label="🔑 Gemini API Key", type="password", placeholder="Paste API Key here")
        model_dropdown = gr.Dropdown(choices=["gemini-2.5-flash", "gemini-2.0-flash"], value="gemini-2.5-flash", label="Model")

    with gr.Row():
        with gr.Column():
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("⚡ Ingest & Process Document", variant="primary")
            status_bar = gr.Textbox(label="Status & Execution Time", interactive=False)
            meta_out = gr.Code(label="Metadata (JSON)", language="json")

        with gr.Column():
            markdown_out = gr.Markdown(label="Extracted Structured Markdown")

    gr.Markdown("### 💬 Hybrid RAG Q&A")
    with gr.Row():
        query_input = gr.Textbox(label="Ask a question about the document", scale=4)
        query_btn = gr.Button("🔍 Search & Answer", variant="secondary", scale=1)

    rag_output = gr.Markdown(label="Grounded Answer")

    process_btn.click(
        fn=process_pdf,
        inputs=[pdf_input, api_key_input, model_dropdown],
        outputs=[markdown_out, meta_out, status_bar]
    )

    query_btn.click(
        fn=answer_rag_query,
        inputs=[query_input],
        outputs=[rag_output]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_5262/2305521637.py:196: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="DocuMind AI") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3ce9ccf05d1800539f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
